# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata.to_json()

print(f"Dataset Title: {metadata.get('name','')}")
print(f"Description: {metadata.get('description','')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the dataset API to enumerate record sets and their fields, referencing them only by their `@id`.

In [ ]:
# List all record set @ids and their field @ids

record_sets = dataset.record_sets
print("Available Record Sets and Their Fields by @id:")
record_sets_ids = []
fields_by_record_set_id = {}
for rs in record_sets:
    rs_id = rs['@id']
    record_sets_ids.append(rs_id)
    print(f"Record Set @id: {rs_id}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    field_ids = [f['@id'] for f in fields]
    fields_by_record_set_id[rs_id] = field_ids
    print(f"  Fields @ids: {field_ids}\n")

# For demonstration: print first record from the first record set
if record_sets_ids:
    sample_rs_id = record_sets_ids[0]
    for rec in dataset.records(record_set=sample_rs_id):
        print(f"Sample record from Record Set {sample_rs_id}:")
        print(rec)
        break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into pandas DataFrames, referencing by @id
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Columns for Record Set {record_set_id}: {dataframes[record_set_id].columns.tolist()}")

# Display head of first dataframe
main_record_set_id = record_sets_ids[0] if record_sets_ids else None
if main_record_set_id:
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This demo will:
  - Filter by a numeric field (e.g., PHQ-9 score).
  - Normalize that field.
  - Group by a demographic attribute (e.g., gender).

In [ ]:
# Locate numeric and grouping fields by their @id

main_rs_fields = fields_by_record_set_id[main_record_set_id]
# Example: find PHQ-9 score field @id and gender field @id
phq9_field_id = None
gender_field_id = None
for fid in main_rs_fields:
    if 'phq' in fid.lower():
        phq9_field_id = fid
    if 'gender' in fid.lower():
        gender_field_id = fid
    if phq9_field_id and gender_field_id:
        break

print(f"PHQ-9 Score Field @id: {phq9_field_id}")
print(f"Gender Field @id: {gender_field_id}")

# Retrieve DataFrame
df = dataframes[main_record_set_id]

# Check for missing values and filter
if phq9_field_id and phq9_field_id in df.columns:
    numeric_field = phq9_field_id
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping
    group_field = gender_field_id
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize distribution of PHQ-9 scores
if phq9_field_id and phq9_field_id in df.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(df[phq9_field_id], bins=20, kde=True)
    plt.title('Distribution of PHQ-9 Scores')
    plt.xlabel('PHQ-9 Score')
    plt.ylabel('Frequency')
    plt.show()

# Boxplot of PHQ-9 scores grouped by gender
if gender_field_id and gender_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df[gender_field_id], y=df[phq9_field_id])
    plt.title('PHQ-9 Score by Gender')
    plt.xlabel('Gender')
    plt.ylabel('PHQ-9 Score')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated loading a FAIR² Mental Health Survey dataset from Kilifi County, Kenya using `mlcroissant`.
- We referenced record sets, fields, and columns by their Croissant `@id` throughout.
- Data was filtered, normalized, grouped, and visualized for key indicators (PHQ-9 scores, gender).
- The dataset is AI-ready and provides rich metadata and survey responses for mental health research and community strategy.

<br>
_End of notebook._